# 11 하이브리드 수요예측 — SBC vs ML

2-type(**고변동 E · 저변동 C**)에 대해, 10장과 같은 **LSTM + Best 임베딩** 예측으로
**SBC vs ML(TS2Vec+KMeans)** 을 제품수 가중 WMAPE로 비교합니다.

논문에서는 저변동(Center A)→ML, 고변동(Center B)→SBC가 유리했습니다.
본 실습의 C(System CV 0.259)·E(0.429)에서도 같은 대응이 나오는지 봅니다.

WMAPE = Σ n_k·MAPE_k / Σ n_k


### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.experiment_data import load_forecast_frames

df, feat_df = load_forecast_frames()
phase2 = pd.read_parquet(DATA_PROCESSED / 'phase2_results.parquet')
phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
print('Phase2 (XGBoost×6임베딩) | rows:', len(phase2), '| 조건:', len(phase2_best))


Phase2 (XGBoost×6임베딩) | rows: 792 | 조건: 12


### ① 가중 WMAPE — type별 SBC vs ML

In [2]:
import numpy as np
from utils.phase_analysis import validation_weights, build_family_from_phase2_best, thesis_wmape_by_type
from utils.stats_summary import type_variation_table
from utils.config import selected_type_list

# 조건별 XGBoost + Best 임베딩(10장) 제품 단위 결과
val_weights = validation_weights(df)
family_final = build_family_from_phase2_best(phase2, phase2_best, val_weights)

# 논문식 WMAPE = Σ n_k·MAPE_k / Σ n_k (클러스터 제품수 가중, §4.5 Eq.50)
type_wmape = thesis_wmape_by_type(family_final)
print('=== type별 SBC vs ML (제품수 가중 WMAPE) ===')
display(type_wmape)
print('scheme 우세:', type_wmape['better_scheme'].value_counts().to_dict())

# System-Level CV(고/저변동 판별 지표)와 대조
var = type_variation_table(df)[['sys_cv', 'sku_mean_cv']].round(3)
sel = selected_type_list()  # [고변동, 저변동]
out = type_wmape.join(var)
out['variation'] = np.where(out.index == sel[0], 'high(고변동)',
                    np.where(out.index == sel[1], 'low(저변동)', ''))
print('=== scheme 우세 × System-Level CV ===')
display(out[['variation', 'sys_cv', 'SBC_wmape', 'ML_wmape', 'better_scheme']])

=== type별 SBC vs ML (제품수 가중 WMAPE) ===


,SBC_wmape,ML_wmape,delta_SBC_minus_ML,better_scheme
type,,,,
C,38.18,38.19,-0.01,SBC
E,46.72,47.60,-0.88,SBC


scheme 우세: {'SBC': 2}
=== scheme 우세 × System-Level CV ===


,variation,sys_cv,SBC_wmape,ML_wmape,better_scheme
type,,,,,
C,low(저변동),0.259,38.18,38.19,SBC
E,high(고변동),0.429,46.72,47.60,SBC


### ② 해석

| type | System CV | SBC WMAPE | ML WMAPE | 우세 |
|------|-----------|-----------|----------|------|
| **C** (저변동) | 0.259 | **38.18** | 38.19 | SBC (−0.01, 동률에 가깝음) |
| **E** (고변동) | 0.429 | **46.72** | 47.60 | SBC (−0.88) |

둘 다 SBC가 앞섭니다. 고변동 E는 논문과 방향이 같고, 저변동 C→ML은 재현되지 않았습니다.

이유는 ML 군집이 **C 30:3 · E 31:2**로 치우쳐 세분화가 거의 안 되기 때문입니다.
SBC는 ADI·CV² 4분류라 계열이 적어도 유형 구분이 유지됩니다.

참고로 type 간 CV 차이(0.259 vs 0.429)도 논문·다이소 실증만큼 크지 않습니다.
변동 구조가 선명하지 않으면 scheme 간 WMAPE 차이도 작아질 수 있습니다.
